In [ ]:
# === PARAMETERS (injected by papermill) ===

# Environment
WIDTH = 9
HEIGHT = 9
NUM_AGENTS = 4

# Experiment Settings
TRAIN_SPLIT = 0.8  # Fraction of positions for training (1.0 = sanity check)
HIDDEN_DIM = 512
NUM_LAYERS = 4

# Training
LEARNING_RATE = 0.0001
BATCH_SIZE = 256
NUM_ITERATIONS = 50000 #50000

# Data Generation
SAMPLES_PER_POSITION = 50
SEED = 42

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
def state_to_mlp_input(apples: np.ndarray, agents: np.ndarray, self_pos: tuple) -> np.ndarray:
    """Convert state to flattened MLP input.
    
    Channels:
        0: apples
        1: other agents (all agents - self)
        2: self (one-hot)
    """
    apples_map = apples.astype(np.float32)
    agents_map = agents.astype(np.float32)
    
    self_map = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    self_map[self_pos[0], self_pos[1]] = 1.0
    
    others_map = agents_map - self_map
    
    return np.stack([apples_map, others_map, self_map], axis=0).flatten()

In [ ]:
def generate_picker_state(picker_pos: tuple):
    """Generate state where self picks apple at picker_pos."""
    apples = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    agents = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    
    # Self on apple
    apples[picker_pos] = 1
    agents[picker_pos] = 1
    
    # Place other agents randomly (not on apple)
    for _ in range(NUM_AGENTS - 1):
        while True:
            pos = (np.random.randint(HEIGHT), np.random.randint(WIDTH))
            if pos != picker_pos:
                agents[pos] += 1
                break
    
    return apples, agents, picker_pos, -1.0


def generate_bystander_state(apple_pos: tuple):
    """Generate state where another agent picks at apple_pos, self is elsewhere."""
    apples = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    agents = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    
    # Apple with another agent on it
    apples[apple_pos] = 1
    agents[apple_pos] = 1
    
    # Self at different position
    while True:
        self_pos = (np.random.randint(HEIGHT), np.random.randint(WIDTH))
        if self_pos != apple_pos:
            agents[self_pos] += 1
            break
    
    # Remaining agents
    placed = 2
    while placed < NUM_AGENTS:
        pos = (np.random.randint(HEIGHT), np.random.randint(WIDTH))
        if pos != apple_pos:
            agents[pos] += 1
            placed += 1
    
    bystander_reward = 2.0 / (NUM_AGENTS - 1)
    return apples, agents, self_pos, bystander_reward


def generate_zero_state(self_pos: tuple):
    """Generate state where no agent is on any apple."""
    apples = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    agents = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
    
    # Self
    agents[self_pos] = 1
    
    # Other agents
    for _ in range(NUM_AGENTS - 1):
        pos = (np.random.randint(HEIGHT), np.random.randint(WIDTH))
        agents[pos] += 1
    
    # Apple NOT on any agent
    while True:
        apple_pos = (np.random.randint(HEIGHT), np.random.randint(WIDTH))
        if agents[apple_pos] == 0:
            apples[apple_pos] = 1
            break
    
    return apples, agents, self_pos, 0.0


def generate_data_for_positions(positions: list, samples_per_position: int):
    """Generate picker, bystander, and zero states for given positions."""
    data = []
    
    for pos in positions:
        for _ in range(samples_per_position):
            apples, agents, self_pos, r = generate_picker_state(pos)
            data.append((apples.copy(), agents.copy(), self_pos, r))
        
        for _ in range(samples_per_position):
            apples, agents, self_pos, r = generate_bystander_state(pos)
            data.append((apples.copy(), agents.copy(), self_pos, r))
        
        for _ in range(samples_per_position):
            apples, agents, self_pos, r = generate_zero_state(pos)
            data.append((apples.copy(), agents.copy(), self_pos, r))
    
    return data

In [ ]:
# All 81 positions
all_positions = [(r, c) for r in range(HEIGHT) for c in range(WIDTH)]

# Shuffle and split
rng = np.random.RandomState(SEED)
rng.shuffle(all_positions)

n_train = int(len(all_positions) * TRAIN_SPLIT)
train_positions = all_positions[:n_train]
test_positions = all_positions[n_train:] if TRAIN_SPLIT < 1.0 else all_positions

print(f"Train split: {TRAIN_SPLIT * 100:.0f}%")
print(f"Train positions: {len(train_positions)}")
print(f"Test positions: {len(test_positions)}")

In [ ]:
print("\nGenerating training data...")
train_data = generate_data_for_positions(train_positions, SAMPLES_PER_POSITION)
print(f"Training samples: {len(train_data)}")

print("\nGenerating test data...")
test_data = generate_data_for_positions(test_positions, SAMPLES_PER_POSITION)
print(f"Test samples: {len(test_data)}")

In [ ]:
class RewardMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

input_dim = 3 * HEIGHT * WIDTH
model = RewardMLP(input_dim, HIDDEN_DIM, NUM_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: hidden_dim={HIDDEN_DIM}, num_layers={NUM_LAYERS}")
print(f"Parameters: {n_params:,}")

In [ ]:
losses = []
model.train()

print(f"\nTraining for {NUM_ITERATIONS} iterations...")
for i in tqdm(range(NUM_ITERATIONS)):
    batch = random.sample(train_data, min(BATCH_SIZE, len(train_data)))
    
    inputs = [state_to_mlp_input(ap, ag, pos) for ap, ag, pos, r in batch]
    targets = [r for ap, ag, pos, r in batch]
    
    x = torch.tensor(np.stack(inputs), dtype=torch.float32).to(device)
    y = torch.tensor(targets, dtype=torch.float32).to(device)
    
    optimizer.zero_grad()
    preds = model(x).squeeze(1)
    loss = nn.functional.mse_loss(preds, y)
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.yscale('log')
plt.title("Training Loss")
plt.xlabel("Iteration")
plt.ylabel("MSE Loss")
plt.savefig("training_loss.png", dpi=100)
plt.show()

In [ ]:
def evaluate_model(model, data, name=""):
    """Evaluate model and return metrics by reward type."""
    model.eval()
    
    if len(data) == 0:
        print(f"[{name}] No data")
        return {}
    
    inputs = [state_to_mlp_input(ap, ag, pos) for ap, ag, pos, r in data]
    targets = np.array([r for ap, ag, pos, r in data])
    
    x = torch.tensor(np.stack(inputs), dtype=torch.float32).to(device)
    
    with torch.no_grad():
        preds = model(x).squeeze(1).cpu().numpy()
    
    results = {}
    
    # Picker
    mask = targets == -1.0
    if mask.sum() > 0:
        mae = np.mean(np.abs(preds[mask] - targets[mask]))
        mape = np.mean(np.abs((preds[mask] - targets[mask]) / targets[mask])) * 100
        results["picker_mae"] = mae
        results["picker_mape"] = mape
        print(f"[{name}] Picker:    MAE={mae:.5f}, MAPE={mape:.2f}%")
    
    # Bystander
    mask = targets > 0
    if mask.sum() > 0:
        mae = np.mean(np.abs(preds[mask] - targets[mask]))
        mape = np.mean(np.abs((preds[mask] - targets[mask]) / targets[mask])) * 100
        results["bystander_mae"] = mae
        results["bystander_mape"] = mape
        print(f"[{name}] Bystander: MAE={mae:.5f}, MAPE={mape:.2f}%")
    
    # Zero
    mask = targets == 0.0
    if mask.sum() > 0:
        mae = np.mean(np.abs(preds[mask] - targets[mask]))
        results["zero_mae"] = mae
        print(f"[{name}] Zero:      MAE={mae:.5f}")
    
    return results

In [ ]:
def evaluate_model(model, data, name=""):
    """Evaluate model and return metrics by reward type."""
    model.eval()
    
    if len(data) == 0:
        print(f"[{name}] No data")
        return {}
    
    inputs = [state_to_mlp_input(ap, ag, pos) for ap, ag, pos, r in data]
    targets = np.array([r for ap, ag, pos, r in data])
    
    x = torch.tensor(np.stack(inputs), dtype=torch.float32).to(device)
    
    with torch.no_grad():
        preds = model(x).squeeze(1).cpu().numpy()
    
    results = {}
    
    # Picker
    mask = targets == -1.0
    if mask.sum() > 0:
        mae = np.mean(np.abs(preds[mask] - targets[mask]))
        mape = np.mean(np.abs((preds[mask] - targets[mask]) / targets[mask])) * 100
        results["picker_mae"] = mae
        results["picker_mape"] = mape
        print(f"[{name}] Picker:    MAE={mae:.5f}, MAPE={mape:.2f}%")
    
    # Bystander
    mask = targets > 0
    if mask.sum() > 0:
        mae = np.mean(np.abs(preds[mask] - targets[mask]))
        mape = np.mean(np.abs((preds[mask] - targets[mask]) / targets[mask])) * 100
        results["bystander_mae"] = mae
        results["bystander_mape"] = mape
        print(f"[{name}] Bystander: MAE={mae:.5f}, MAPE={mape:.2f}%")
    
    # Zero
    mask = targets == 0.0
    if mask.sum() > 0:
        mae = np.mean(np.abs(preds[mask] - targets[mask]))
        results["zero_mae"] = mae
        print(f"[{name}] Zero:      MAE={mae:.5f}")
    
    return results

In [ ]:
print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Train Split: {TRAIN_SPLIT * 100:.0f}%")
print(f"Model: {HIDDEN_DIM} hidden x {NUM_LAYERS} layers")
print("=" * 60)

print("\n--- TRAIN SET (positions seen during training) ---")
train_results = evaluate_model(model, train_data, "TRAIN")

print("\n--- TEST SET (positions NOT seen during training) ---")
test_results = evaluate_model(model, test_data, "TEST")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Train Picker MAPE:    {train_results.get('picker_mape', 'N/A'):.2f}%")
print(f"Test Picker MAPE:     {test_results.get('picker_mape', 'N/A'):.2f}%")
print(f"Train Bystander MAPE: {train_results.get('bystander_mape', 'N/A'):.2f}%")
print(f"Test Bystander MAPE:  {test_results.get('bystander_mape', 'N/A'):.2f}%")

if TRAIN_SPLIT < 1.0:
    picker_gap = test_results.get('picker_mape', 0) - train_results.get('picker_mape', 0)
    bystander_gap = test_results.get('bystander_mape', 0) - train_results.get('bystander_mape', 0)
    print(f"\nGeneralization Gap (Test - Train):")
    print(f"  Picker:    {picker_gap:+.2f}%")
    print(f"  Bystander: {bystander_gap:+.2f}%")